In [56]:
#Use the Directory Loader to Load the Data

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Load data
loader = DirectoryLoader('PDF_Inprogress', glob="**/*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()

#Split (Chunk) data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)


In [57]:
len(chunks)

13529

In [58]:
chunks[:10]

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2010-04-14T09:10:40-04:00', 'source': 'PDF_Inprogress/bls_2800_2010_libraryedition.pdf', 'file_path': 'PDF_Inprogress/bls_2800_2010_libraryedition.pdf', 'total_pages': 890, 'format': 'PDF 1.4', 'title': '2010-11 Occupational Outlook Handbook', 'author': 'U.S. Bureau of Labor Statistics', 'subject': '', 'keywords': '', 'moddate': '2019-04-01T12:14:24-05:00', 'trapped': '', 'modDate': "D:20190401121424-05'00'", 'creationDate': "D:20100414091040-04'00'", 'page': 0}, page_content='Occupational\t\n2010-11\nOutlook\t\nLibrary Edition\nHandbook\nU.S. Department of Labor\nHilda L. Solis, Secretary\nU.S. Bureau of Labor Statistics\nKeith Hall, Commissioner\nJanuary 2010\nBulletin 2800\nSuggested citation: U.S. Bureau of Labor Statistics, U.S. Department of Labor, Occupational Outlook Handbook, 2010-11 \nLibrary Edition, Bulletin 2800. Superintendent of Documents, U.S. Government Prin

In [49]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [12]:
# embeddings = OllamaEmbeddings(model="bge-m3")
embeddings = OllamaEmbeddings(model="embeddinggemma:300m")

In [ ]:
#Run this for first time, then if want to add more chunks embedding use code in below cell
print('Storing in Chroma using Ollama')
Vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory = './ollama_chroma_db2'
    
)

In [19]:
# Run this code if you want to add new documnts chunks to the existing database otherwise comment it
Vector_db = Chroma(
    persist_directory='./ollama_chroma_db2',
    embedding_function=embeddings
 )

Vector_db.add_documents(documents=chunks)

In [ ]:
#Define a safe batch size (5000 is the industry standard for Chroma)
batch_size = 5000 

#Iterate through your chunks in steps of 5000
for i in range(0, len(chunks), batch_size):
    # Slice the list to get the current batch
    current_batch = chunks[i : i + batch_size]
    
    print(f"Adding batch {i//batch_size + 1} ({len(current_batch)} chunks)...")
    
    #Add this specific batch to the existing database
    Vector_db.add_documents(documents=current_batch)

print("✅ Successfully added all chunks to the database!")

In [59]:
from tqdm import tqdm

batch_size = 5000 

for i in tqdm(range(0, len(chunks), batch_size), desc="Embedding Chunks"):
    current_batch = chunks[i : i + batch_size]
    Vector_db.add_documents(documents=current_batch)

Embedding Chunks: 100%|██████████| 3/3 [10:24<00:00, 208.08s/it]


In [60]:
count = Vector_db._collection.count()
print(f"Total chunks in database: {count}")

Total chunks in database: 230924


In [14]:
import ollama

In [4]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import ChatPromptTemplate

In [69]:
# Initialize the LLM via Ollama
# Make sure to run 'ollama pull llama3.2' in your terminal first
llm = ChatOllama(model="llama3.2", temperature=0.4)
# llm = ChatOllama(model="phi4-mini", temperature=0.3)
# llm = ChatOllama(model="deepseek-r1:1.5b", temperature=0.3)

# Create a System Prompt
# This tells the LLM to ONLY use the PDFs to answer
system_prompt = (
    "Role: You are an expert People Analytics Consultant and Labor Historian specializing in the BLS Occupational Outlook Handbooks (OOH) from 1949 to 2024."
    "Task: Your goal is to extract, analyze, and synthesize workforce intelligence to identify long-term patterns in job roles, skills, and economic shifts."
    "Strict Grounding: Base your answers ONLY on the provided PDF context. If the information is not present in the documents, state:I do not have sufficient historical data in the uploaded handbooks to answer this."
    "Handling Time-Series Data: When asked about a job role, always specify the Handbook Year you are referencing. If multiple years are available, provide a comparative timeline."
    "Key Pillars of Analysis: For every job role, prioritize extracting:Entry Barriers: Educational and training requirements.Technological Drivers: Mention of new tools (e.g., calculators, computers, AI).Economic Outlook: Projected growth or decline as stated by the BLS at that time."
    "Output Structure: Use Markdown tables for comparisons. Use bullet points for Key Insights and Skill Gaps."
    "Constraint: Never use outside general knowledge to fill in the blanks of a specific handbook's data."
    "Linguistic Sensitivity: Recognize that terminology changes. (e.g., Draftsmen in 1950 vs CAD Operators in 2020). Treat these as a single professional evolution."
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Create the RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(Vector_db.as_retriever(search_kwargs={"k": 10}), question_answer_chain)

# Ask a Question
user_query = input('Ask me a question: ')
response = rag_chain.invoke({"input": user_query})

print(f"\n🔍 Context Source: {response['context'][0].metadata['source']}")
print(f"🤖 Answer: {response['answer']}")



🔍 Context Source: PDF_Inprogress/BLS 1965-pt2.pdf
🤖 Answer: According to the BLS Occupational Outlook Handbook (BLS_1650_1970_pt2-compressed.pdf), here are the earnings and working conditions for Photographers in 1968:

**Earnings:**

* Beginning photographers generally earned from $85 to $105 a week.
* Experienced newspaper photographers on large dailies earned up to $235 a week.

**Working Conditions:**

* Approximately 60,000 photographers were employed in 1968.
* About half of them worked in portrait or commercial studios, many as salaried employees.
* Sizable numbers were employed in industry, some working for Federal, State, and local government agencies.
* Others operated camera stores or worked on the staffs of newspapers and magazines.
* Still others worked as freelance photographers.

Note: The provided answer is based solely on the information available in the BLS_1650_1970_pt2-compressed.pdf document.


In [80]:
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Define the Contextualization Prompt
# This "pre-processes" the user's question to include history before searching the PDFs
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Create the history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, Vector_db.as_retriever(search_kwargs={"k": 6}), contextualize_q_prompt
)

#  Update your original System Prompt to accept history
# (Added MessagesPlaceholder for chat_history)
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt), # Your existing detailed system_prompt string
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# Build the Final RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# The Conversation Loop
chat_history = []  # This list stores the conversation state

while True:
    user_query = input('\nAsk me a question (or type "exit"): ')
    if user_query.lower() == "exit": 
        break

    # We pass the 'chat_history' list into the invoke call
    response = rag_chain.invoke({"input": user_query, "chat_history": chat_history})
    
    print(f"\n🔍 Context Source: {response['context'][0].metadata['source']}")
    print(f"\n🤖 Answer: {response['answer']}")
    
    # Update history with the latest exchange to keep the "memory" alive
    chat_history.extend([
        HumanMessage(content=user_query),
        AIMessage(content=response["answer"]),
    ])
    
    # Optional: Keep only the last 5 exchanges to save memory/context window
    chat_history = chat_history[-5:]


🔍 Context Source: PDF_Inprogress/BLS 1965-pt1.pdf

🤖 Answer: I don't have sufficient historical data in the uploaded handbooks to answer this question specifically about future job roles and dominant skills. However, I can suggest that the BLS Occupational Outlook Handbooks from 1949 to 2024 provide information on emerging trends and skills in various occupations.

To answer your question, I would recommend analyzing the charts and tables related to emerging occupations and technological drivers mentioned in the handbooks. For example, Chart 8 mentions "Future Working Population" but does not specifically list dominant skills for future job roles.

If you'd like, I can help you explore other sections of the handbook or provide general insights on workforce trends based on historical data.

🔍 Context Source: PDF_Inprogress/BLS 1961-pt1.pdf

🤖 Answer: **Impact on Demand:**
Changes in one job category can impact demand in another through various mechanisms. For example:

* Automation in 